# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a walkthrough for accessing and exploring the FAIR^2 dataset using the `mlcroissant` library, following best practices for referencing entities by their Croissant `@id` fields.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields using their @id
if not hasattr(metadata, 'record_sets') or not metadata.record_sets:
    print("No record sets found in the metadata.")
else:
    print('Available record sets and fields:')
    for rs in metadata.record_sets:
        print(f"\nRecord set: {rs['@id']} (name: {rs.get('name', '')})")
        if 'fields' in rs:
            for field in rs['fields']:
                print(f"  Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")

In [ ]:
# If record sets are available, preview records from the first record set
if hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_set_id = metadata.record_sets[0]['@id']
    print(f"\nPreviewing records from record set with @id: {record_set_id}")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets available to preview records.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
dataframes = {}

# Extract all record_set @ids
record_set_ids = [rs['@id'] for rs in getattr(metadata, 'record_sets', [])]

if not record_set_ids:
    print("No record sets available for extraction.")
else:
    for rid in record_set_ids:
        print(f"Loading record set: {rid}")
        records = list(dataset.records(record_set=rid))
        if not records:
            print(f"No records found for record set {rid}.")
            continue
        dataframes[rid] = pd.DataFrame(records)

    # For demonstration, pick the first record set
    active_rid = record_set_ids[0]
    if active_rid in dataframes:
        df = dataframes[active_rid]
        print(f"Columns in record set {active_rid}:")
        print(df.columns.tolist())
        print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, you will see how to select and analyze a numeric field and perform group-by operations. Fields and columns must be referenced by their `@id`.

In [ ]:
# Choose the first record set and first numeric field for EDA

# Find a numeric field in the first record set
numeric_field_id = None
group_field_id = None

if hasattr(metadata, 'record_sets') and metadata.record_sets:
    rs = metadata.record_sets[0]
    for field in rs.get('fields', []):
        if field.get('dataType', '').lower() in ['float', 'integer', 'number']:
            numeric_field_id = field['@id']
            break
    # Try to find a group field (categorical)
    for field in rs.get('fields', []):
        if field.get('dataType', '').lower() in ['text', 'string', 'boolean', 'category']:
            group_field_id = field['@id']
            break

active_rid = metadata.record_sets[0]['@id'] if hasattr(metadata, 'record_sets') and metadata.record_sets else None
if not numeric_field_id or not active_rid or active_rid not in dataframes:
    print("Cannot find a suitable numeric field or record set for EDA.")
else:
    df = dataframes[active_rid]
    field_name = numeric_field_id if numeric_field_id in df.columns else df.columns[0]

    # Filter for values above a threshold
    threshold = 10
    filtered_df = df[df[field_name] > threshold]
    print(f"Filtered records where {field_name} > {threshold}:")
    print(filtered_df.head())

    # Normalize this field
    norm_col = f"{field_name}_normalized"
    filtered_df[norm_col] = (filtered_df[field_name] - filtered_df[field_name].mean()) / filtered_df[field_name].std()
    print(f"\nNormalized {field_name} for filtered records:")
    print(filtered_df[[field_name, norm_col]].head())

    if group_field_id and group_field_id in df.columns:
        # Only calculate mean for numeric columns
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped mean values by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution and relationships for the selected numeric field
if not numeric_field_id or not active_rid or active_rid not in dataframes:
    print("No numeric field or dataframe available for visualization.")
else:
    df = dataframes[active_rid]
    field_name = numeric_field_id if numeric_field_id in df.columns else df.columns[0]

    plt.figure(figsize=(8, 4))
    sns.histplot(df[field_name].dropna(), kde=True)
    plt.title(f"Distribution of {field_name}")
    plt.xlabel(field_name)
    plt.ylabel('Count')
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field_id, y=field_name, data=df)
        plt.title(f"{field_name} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and examined the FAIR^2 dataset on predictors for adoption of indigenous and modern knowledge in rangeland management. Using the `mlcroissant` library, we identified record sets and fields by their `@id`s, extracted data, applied filters and normalization, and visualized key distributions and group differences. This workflow demonstrates reproducible, schema-aware data exploration that can be adapted to new Croissant-compliant datasets.